# 03 - Backbone Comparison

**Purpose.** Compare individual CNN backbones before evaluating the ensemble. A strong ensemble claim needs transparent base-model behavior.

**Research integrity rule.** Report individual models even when the ensemble is the headline result.

In [ ]:
from pathlib import Path
import json

import pandas as pd

PROJECT = Path('..').resolve()
EXPERIMENT = PROJECT / 'artifacts' / 'paper_2022_idc' / 'experiment.json'

# Override this if you ran a different output directory.
if not EXPERIMENT.exists():
    EXPERIMENT = PROJECT / 'artifacts' / 'simple_real_test' / 'experiment.json'

print(EXPERIMENT)
print('exists:', EXPERIMENT.exists())

In [ ]:
if not EXPERIMENT.exists():
    print('Run dcpgann-train first, then return to this notebook.')
else:
    report = json.loads(EXPERIMENT.read_text())
    print('backbones:', report['backbones'])
    print('split sizes:', report['split_sizes'])

## Model Size and Trainable Parameters

In [ ]:
if EXPERIMENT.exists():
    sizes = pd.DataFrame(report['model_summaries']).T
    sizes['trainable_fraction'] = sizes['trainable'] / sizes['total']
    display(sizes.sort_index())

## Individual Test Metrics

These numbers show what each backbone contributes before optimization.

In [ ]:
if EXPERIMENT.exists():
    metrics = pd.DataFrame(report['individual_test_metrics']).T
    cols = ['accuracy', 'balanced_accuracy', 'precision', 'recall_sensitivity', 'specificity', 'f1', 'roc_auc']
    display(metrics[cols].sort_values('balanced_accuracy', ascending=False))

## Training Histories

If the history is noisy or validation performance collapses, the ensemble can hide a training problem. Inspect this before interpreting final scores.

In [ ]:
if EXPERIMENT.exists():
    rows = []
    for model, history in report['train_histories'].items():
        for row in history:
            rows.append({'model': model, **row})
    history_df = pd.DataFrame(rows)
    if len(history_df):
        display(history_df.groupby('model').tail(3))
    else:
        print('No training history found.')